# Part 4: Choosing the Right Architecture
## Content Creation Studio Workshop

We've built individual agents (Parts 1–3). Before writing any more agent code, we need to answer the most important question: **what pattern fits your problem?**

Using the wrong architecture leads to brittle systems, unnecessary cost, and agents that fight each other instead of collaborating.

> This notebook is based on the official Google Cloud guidance: [Choose a design pattern for your agentic AI system](https://cloud.google.com/architecture/choose-design-pattern-agentic-ai-system).

---

## The Content Creation Studio Playbook Series
- Part 1: Building Your First AI Agent ✅
- Part 2: Giving Your Agent Custom Tools ✅
- Part 3: Building Agent Teams with Agent-as-a-Tool ✅
- **Part 4: Choosing the Right Architecture** (You are here) 👈
- Part 5: Sequential Workflows
- Part 6: Building Iterative Workflows with LoopAgent
- Part 7: Efficient Workflows with ParallelAgent
- Part 8: Callbacks, Context & Memory
- Part 9: The Capstone Project
- Part 10: Deploying to Vertex AI Agent Engine

---

## What You'll Learn

- The four pattern categories from Google Cloud's agentic AI design guidance
- The core distinction between **LLM-Driven** and **Workflow-Driven** architectures
- The constraints that should drive your pattern choice (cost, determinism, safety)
- How the Content Creation Studio workshop maps to each pattern

## 1. ADK Agent Type Hierarchy

The official ADK documentation categorizes agents into three types:

![ADK Agent Types](https://google.github.io/adk-docs/assets/agent-types.png)

*Source: [Google ADK Agents Documentation](https://google.github.io/adk-docs/agents/)*

| Type | Examples | Who controls routing? |
|---|---|---|
| **LLM Agent** | `Agent` | The LLM decides at runtime |
| **Workflow Agent** | `SequentialAgent`, `LoopAgent`, `ParallelAgent` | Your code (deterministic) |
| **Custom Agent** | Subclass of `BaseAgent` | You implement the logic |

Parts 5–7 of this workshop cover each workflow agent type in depth. This notebook helps you understand **when** to reach for each one.

## 2. Pattern Categories

Google Cloud groups agentic patterns into four categories based on your workload's characteristics.

---

### Deterministic Workflows
Execution order is fixed in code — no model decides the routing.

| If your task looks like this… | Pattern | ADK type |
|---|---|---|
| Multi-step pipeline where each step depends on the previous | **Sequential** | `SequentialAgent` |
| Independent sub-tasks that can run at the same time | **Parallel** | `ParallelAgent` |
| Complex generation requiring progressive quality improvement | **Iterative Refinement** | `LoopAgent` |

---

### Dynamic Orchestration
A model decides routing and delegation at runtime.

| If your task looks like this… | Pattern | ADK type |
|---|---|---|
| Varied inputs needing adaptive routing to specialized agents | **Coordinator** | `Agent` with `sub_agents` |
| Complex, ambiguous tasks requiring multi-level decomposition | **Hierarchical Task Decomposition** | Nested `Agent` hierarchy |

---

### Iterative Workflows
Repeat execution until a condition is met.

| If your task looks like this… | Pattern | ADK type |
|---|---|---|
| Automated repetition with a predefined termination condition | **Loop** | `LoopAgent` |
| Output requires distinct validation before completion | **Review and Critique** | `LoopAgent` with checker + improver |

---

### Special Requirements

| If your task looks like this… | Pattern |
|---|---|
| High-stakes decisions requiring human judgment or approval | **Human-in-the-Loop** |

> **Start simple:** A non-agentic solution (a single LLM call) may be more cost-effective for predictable tasks like summarization or classification. Only add agents when the task genuinely requires multi-step reasoning, iteration, or parallelism.

## 3. The Core Distinction: LLM-Driven vs Workflow-Driven

All patterns reduce to one key architectural question: **who controls the flow — the LLM or your code?**

| | **LLM-Driven (`Agent`)** | **Workflow-Driven (`SequentialAgent`, etc.)** |
|---|---|---|
| **Routing** | LLM decides what to call | Execution order fixed in code |
| **Flexibility** | Handles any request dynamically | Fixed pipeline for specific tasks |
| **Predictability** | Variable (LLM may choose differently each run) | Deterministic (same path every run) |
| **Cost** | Each routing decision = 1 LLM call | Zero LLM calls for routing |
| **Best For** | Orchestrators, chatbots, open-ended tasks | Pipelines, content generation, batch processing |

### When to Choose Which

Use **LLM-Driven** when:
- User intent is unpredictable (chatbot, assistant)
- Multiple possible actions depending on input
- You need the agent to reason about what to do

Use **Workflow-Driven** when:
- Steps are always the same (pipeline)
- Order matters and is known ahead of time
- You want predictable, cost-efficient execution

Use **Hybrid** when:
- Top-level routing needs flexibility → LLM-Driven orchestrator
- Inner execution needs reliability → Workflow-Driven pipelines
- **This is the most common pattern in production systems**

## 4. The Constraints That Drive Your Choice

### Cost and Latency
Every agent hop is an extra LLM call. A 5-agent pipeline uses at least 5× the tokens of a single agent. Parallel agents reduce latency but increase concurrent cost.

**Practical rules:**
- Use deterministic patterns when the steps are well-defined
- Use smaller models for simple sub-agents (summarization, formatting)
- Replace LLM agents with plain Python functions when no reasoning is needed

### Determinism vs Flexibility
Deterministic patterns (Sequential, Parallel, Loop) are faster and more predictable — the code controls the flow. Dynamic patterns (Coordinator, Hierarchical) are more flexible but add routing overhead and are harder to debug.

### Safety and Human Oversight
Fully automated pipelines work well for low-stakes tasks like content generation. For financial, medical, or legal decisions, use Human-in-the-Loop — the agent pauses at checkpoints for human review before acting.

## 5. How This Workshop Maps to the Framework

| Stage | Pattern | Why |
|---|---|---|
| Research → Draft | **Sequential** | Draft always depends on research — fixed order, no model routing needed |
| Draft → Quality check → Improve | **Review and Critique** inside a **Loop** | Critic evaluates, generator revises; repeats until quality threshold met |
| Blog + Social + Email + SEO | **Parallel** | All four are independent — run concurrently to reduce latency |
| Route create vs analyse | **Coordinator** | User intent is open-ended — model decides which pipeline to invoke |

And the LLM-Driven vs Workflow-Driven split:

| Component | Type | Reason |
|---|---|---|
| `master_orchestrator_agent` | **LLM-Driven** | Routes based on open-ended user intent |
| `research_and_draft_workflow` | **Workflow-Driven** | Always: research → draft, fixed order |
| `quality_improvement_loop` | **Workflow-Driven** | Always: check → improve, repeat |
| `parallel_content_creation` | **Workflow-Driven** | Always: 4 channels in parallel |

**The key insight:** Use LLM-Driven agents at the *boundaries* where human intent is ambiguous. Use Workflow-Driven agents *inside* those boundaries where the steps are well-defined.

Keep this table in mind as you build Parts 5–7!

## Recap

| Concept | Key Takeaway |
|---|---|
| Pattern categories | Sequential, Parallel, Loop, Coordinator, Hierarchical, Human-in-the-Loop |
| LLM-Driven | LLM controls routing — flexible, but costly and less predictable |
| Workflow-Driven | Code controls routing — fast, deterministic, zero routing cost |
| Hybrid | LLM-Driven orchestrator + Workflow-Driven pipelines (most common in practice) |
| Start simple | Single LLM call first; add agents only when genuinely needed |

---

## What's Next?

In **Part 5**, we apply the first pattern from this framework: **Sequential Workflows**. You'll build a `SequentialAgent` that runs topic research then content drafting in a fixed order, with automatic state passing between agents.

See you in Part 5! 🚀